# Repack NanoAOD ROOT files over Dask + XRootD

Standalone driver for the `repack_inputs` package. For each chunk:

1. Worker `xrdcp`s the chunk's input files into local scratch.
2. Slices / re-baskets / re-compresses via the vendored `root_repack`.
3. Merges into one local file.
4. `xrdcp`s the merged file straight to the final XRootD URL (no `.tmp + mv`, since most CMS SEs reject user-level rename).
5. Cleans up local scratch on task exit.

Driver-side: load fileset JSON, plan chunks, ship the package via `client.upload_file`, run with a tqdm progress bar, write an output fileset JSON.

## Dask client

In [ ]:
CLIENT_ADDRESS = "tls://localhost:8786"  # Dask scheduler URL

## Configuration

Every CLI flag has a Python equivalent here. `BASKET_SIZE` / `BASKET_SIZES` / `AUTO_FLUSH` trigger a rewrite pass per input (more scratch usage); leave as `None` for a plain merge/split.

In [ ]:
# --- Inputs ---
INPUT_JSON = "~/intc/integration-challenge/cms/example_cms/outputs/metadata/nanoaods.json"

# --- Outputs ---
OUTPUT_DIR_URL = "root://xrootd-local.unl.edu:1094//store/user/IC/repack_out"
OUTPUT_SUBDIR = "{dataset}/{systematic}"  # under OUTPUT_DIR_URL; files become 001.root, 002.root, ...
OUTPUT_FILESET_JSON = "outputs/repack/nanoaods_repacked.json"

# --- Split / merge ---
N_EVENTS = 1_000_000      # None = one output per (dataset, systematic); int = cap per output
EVENT_TREE = "Events"

# --- Tree re-encoding ---
BASKET_SIZE = None         # single size for all branches, e.g. "64k"
BASKET_SIZES = None        # list of patterns, e.g. ["Muon_*=128k", "Jet_*=256k"]
AUTO_FLUSH = None          # e.g. "30M" (byte threshold) or an int (entry count)

# --- TFileMerger options ---
FAST = False               # ROOT fast-merge mode
KEEP = False               # keep input compression instead of re-compressing
SORT = "branch"            # branch | offset | entry
COMPRESS = "same"          # e.g. "zstd=9", "lz4", "same"
IOFEATURES = None          # e.g. ["GenerateOffsetMap"]
VERBOSE = 0

# --- Worker scratch ---
SCRATCH_ROOT = "/tmp/repack_inputs"
MAX_SCRATCH_GB = 7.0       # peak per task ~ inputs + slice-temps + output; stay below worker quota

# --- Runtime ---
OVERWRITE = False          # if True, xrdcp -f overwrites existing outputs on XRootD
PROGRESS = True
RAISE_ON_ERROR = True      # if False, failures are returned as exceptions in the results dict

# --- Dry run ---
DRY_RUN = False            # True = plan + summarize only; skip mkdir, dask, output JSON

## Imports

In [ ]:
import sys
from pathlib import Path

# repack_inputs lives at <repo>/repack_inputs; make sure <repo> is on sys.path.
repo_root = Path.cwd()
if not (repo_root / "repack_inputs").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
from dask.distributed import Client

from repack_inputs import (
    load_fileset,
    plan_chunks,
    prepare_output_dirs,
    run_repack,
    upload_package,
    write_output_fileset,
)

## Load fileset and plan chunks

In [ ]:
files = load_fileset(INPUT_JSON)
ds_syst_pairs = {(f.dataset, f.systematic) for f in files}
print(f"{len(files)} input files across {len(ds_syst_pairs)} dataset/systematic pairs")
print(f"total events: {sum(f.nevts for f in files):,}")

In [ ]:
plans = plan_chunks(
    files,
    output_dir_url=OUTPUT_DIR_URL,
    n_events=N_EVENTS,
    output_subdir=OUTPUT_SUBDIR,
)
print(f"{len(plans)} output chunks planned, {sum(p.total_events for p in plans):,} total events")
for plan in plans[:5]:
    print(f"  {plan.output_url}  ({plan.total_events:,} events from {len(plan.unique_sources)} source files)")
if len(plans) > 5:
    print(f"  ... ({len(plans) - 5} more)")

In [ ]:
from collections import Counter

per_ds = Counter((p.dataset, p.systematic) for p in plans)
event_counts = [p.total_events for p in plans]
segment_counts = [len(p.segments) for p in plans]
source_counts = [len(p.unique_sources) for p in plans]

print(f"chunks: {len(plans)}")
print(f"events per chunk:   min={min(event_counts):,}  max={max(event_counts):,}  mean={sum(event_counts) // len(event_counts):,}")
print(f"segments per chunk: min={min(segment_counts)}  max={max(segment_counts)}")
print(f"sources  per chunk: min={min(source_counts)}  max={max(source_counts)}   (= xrdcp ops per task)")

top = sorted(per_ds.items(), key=lambda kv: -kv[1])[:10]
print("\ntop-10 (dataset, systematic) by chunk count:")
for (ds, syst), n in top:
    print(f"  {ds}/{syst}: {n} chunks")

In [ ]:
if DRY_RUN:
    n_dirs = len({p.output_url.rsplit("/", 1)[0] for p in plans})
    print(f"[dry run] would mkdir -p {n_dirs} output directories on XRootD")
else:
    prepare_output_dirs(plans)
    print(f"mkdir -p done for {len({p.output_url.rsplit('/', 1)[0] for p in plans})} output directories")

## Run the repack

In [ ]:
if DRY_RUN:
    print(f"[dry run] would submit {len(plans)} tasks to {CLIENT_ADDRESS}; skipping")
    results = {}
else:
    with Client(CLIENT_ADDRESS) as client:
        upload_package(client)
        results = run_repack(
            client,
            plans,
            scratch_root=SCRATCH_ROOT,
            max_scratch_gb=MAX_SCRATCH_GB,
            overwrite=OVERWRITE,
            event_tree=EVENT_TREE,
            basket_size=BASKET_SIZE,
            basket_sizes=BASKET_SIZES,
            auto_flush=AUTO_FLUSH,
            fast=FAST,
            keep=KEEP,
            sort=SORT,
            compress=COMPRESS,
            iofeatures=IOFEATURES,
            verbose=VERBOSE,
            progress=PROGRESS,
            raise_on_error=RAISE_ON_ERROR,
        )

    n_ok = sum(1 for v in results.values() if isinstance(v, str))
    n_fail = len(results) - n_ok
    print(f"wrote {n_ok}/{len(results)} outputs ({n_fail} failed)")
    for url, outcome in results.items():
        if not isinstance(outcome, str):
            print(f"  FAIL {url}: {outcome!r}")

## Write output fileset JSON

Same shape as the input JSON, listing only chunks that succeeded. Feed this back into downstream tools.

In [ ]:
if DRY_RUN:
    print(f"[dry run] would write output fileset JSON to {OUTPUT_FILESET_JSON}")
else:
    out_json = write_output_fileset(plans, results, OUTPUT_FILESET_JSON)
    print(f"output fileset JSON: {out_json}")